# Proyecto RAG con LLM y métricas de evaluación

Este notebook implementa un **sistema RAG (Retrieval-Augmented Generation)** que combina:

1. **Carga y procesamiento de documentos** (PDFs reales).
2. **Chunking y generación de embeddings** para cada fragmento.
3. **Búsqueda semántica con FAISS HNSW** y BM25 híbrido.
4. **Reranking con Cross-Encoder** para mejorar precisión.
5. **Generación de respuestas con un LLM (Qwen 4-bit)**.
6. **Validación y métricas**: grounding score, BLEU, ROUGE, y citas por oración.

El objetivo es **responder preguntas usando solo la información contenida en los documentos**, minimizando alucinaciones y asegurando trazabilidad de las respuestas.

### 0. Instalación de librerías
Se instalan todas las dependencias necesarias:
- `transformers`, `accelerate`, `bitsandbytes`, `sentencepiece` → para LLM y cuantización 4-bit.
- `faiss-cpu`, `sentence-transformers`, `datasets` → para embeddings y FAISS.
- `rank_bm25`, `rouge_score`, `nltk`, `pypdf` → para recuperación híbrida, evaluación automática y manejo de PDFs.

Garantiza que todo el pipeline (RAG, reranking, métricas, LLM) pueda ejecutarse sin errores por falta de librerías.

### Instalación de librerías

In [ ]:
!pip install -U transformers accelerate bitsandbytes sentencepiece
!pip install faiss-cpu sentence-transformers datasets
!pip install rank_bm25 rouge_score nltk pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 25.7 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=18caa39f03d7204b61f1806a812344085bd4ebb8e6801b9e52cd8863332a3b4c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


### 1. Preparación de carpeta de documentos
Se crea la carpeta `/content/docs` donde se subirán los PDFs que formarán el dataset. Esto asegura que el pipeline RAG tenga un lugar definido para leer los documentos.

In [ ]:
# comandos para crear un carpeta en los contenidos de nombre "docs", donde almacenaremos los documentos
import os
os.makedirs("/content/docs", exist_ok=True)
print("Carpeta lista — sube tus PDFs a la carpeta docs en el panel de archivos")

Carpeta lista — sube tus PDFs a la carpeta docs en el panel de archivos


### 2. Dataset (Carga de PDFs)
Se leen los PDFs de la carpeta, página por página, limpiando saltos de línea y espacios. Cada página se guarda en `texts` y se le asigna un título en `titles` Esto prepara los documentos para el pipeline RAG. Se imprime la cantidad de páginas cargadas y un ejemplo de contenido.

In [ ]:
from pypdf import PdfReader
import re, os

# creamos dos listas vacias, para almacenar la informacion
texts, titles = [], []

# Listamos todos los archivos dentro de la carpeta /content/docs.
for filename in os.listdir("/content/docs"):
    # Verifica que el archivo termine en .pdf. Si no es PDF, lo salta (continue)
    if not filename.endswith(".pdf"):
        continue
    # Carga el PDF para poder leer sus páginas.
    reader   = PdfReader(f"/content/docs/{filename}")
    # Obtienemos el nombre del archivo sin la extensión .pdf.
    doc_name = filename.replace(".pdf", "")

    # Recorrer páginas del PDF
    # Extraemos el texto de la página, si no hay texto (None) = cadena vacía.
    # eliminamos saltos de línea "\n", reemplazamos espacios por uno solo.
    # strip(): elimina espacios al inicio y final.
    for i, page in enumerate(reader.pages):
        texto = page.extract_text() or ""
        texto = re.sub(r"\s+", " ", texto.replace("\n", " ")).strip()
        if len(texto) > 100:
            texts.append(texto)
            titles.append(f"{doc_name} · p{i+1}")
    #-----------------------------------------------------------

# imprimimos cuantas paginas se cargaron
print("Páginas cargadas:", len(texts))
print("Ejemplo —", titles[0], ":", texts[0][:200])

Páginas cargadas: 850
Ejemplo — oracle-introduccion · p1 : Manual Curso Introductorio a la Administración de Oracle INTRODUCCIÓN A LA ADMINISTRACIÓN DE ORACLE MANUAL DEL CURSO


### 3. Chunking con Overlap

En esta sección se divide el texto en fragmentos (chunks) con solapamiento(overlap) para mantener continuidad entre ellos.

**Tradeoff (equilibrio) del tamaño de chunk**
- Chunk pequeño → búsqueda precisa, pero puede perder contexto
- Chunk grande → más contexto para el LLM, pero menos precisión al buscar

**El Tradeoff (equilibrio) del overlap:**
- Chunk pequeño → búsqueda precisa, pero puede perder contexto
- Chunk grande → más contexto para el LLM, pero menos precisión al buscar

Se utiliza `size=400, overlap=100`  como un equilibrio entre contexto, precisión y eficiencia.

In [ ]:
# Defines una función llamada chunk_text con parametros (text, title, size=400, overlap=100):
# size = tamaño de 400 caracteres
# overlap = solapamiento de 100 caracteres
def chunk_text(text, title, size=400, overlap=100):
    # Lusta vacia para almacenamiento
    chunks = []
    # fijamos un valor (dónde empezar a cortar el texto.)
    start = 0
    # bucle que se ejecuta mientras no haya recorrido toda la longitud de texto
    while start < len(text):
        # Agregamos un nuevo elemento a la lsita chunk y respectivo titulo
        chunks.append({"text": text[start:start+size], "title": title})
        start += size - overlap # <---- aquí se crea el "overlap": cada chunk comparte parte del texto con el anterior
    return chunks
# ------------------------------------------------------------------------------

# definimos una lista vacia
all_chunks = []
# recursivamente agregamos los chunks y sus titulos definidos en la lista all_chunks
for text, title in zip(texts, titles):
    all_chunks.extend(chunk_text(text, title))

# listamos los pedazos de texto con los títulos correspondientes.
chunk_texts  = [c["text"]  for c in all_chunks]
chunk_titles = [c["title"] for c in all_chunks]

# cantidad de chunk generado
print("Total chunks:", len(all_chunks))

Total chunks: 4964


### 4. Embeddings

Se convierten los chunks de texto en vectores numéricos (embeddings) usando un modelo pre-entrenado. Cada vector representa el significado de un chunk, lo que permite comparar y buscar información semántica de manera eficiente. (Textos con significado similar quedan cerca en el espacio vectorial).

In [ ]:
# librería para generar embeddings a partir de texto.
# numpy (np): para manipular vectores/matrices de manera eficiente.
from sentence_transformers import SentenceTransformer
import numpy as np

# Cargamos un modelo pre-entrenado.
embedder   = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Convertimos cada texto de "chunk_texts" en un vector numérico (embedding).
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)
# convertimos la lista de vectores "embeddings" en un array de Numpy.
embeddings = np.array(embeddings).astype("float32")

# mostramos de que dimension es la matriz generada
print("Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Shape: (4964, 384)


### 5. FAISS HNSW
Se utiliza **FAISS** con **HNSW** (Hierarchical Navigable Small World) para indexar los embeddings de los chunks y permitir búsquedas semánticas rápidas.

**HNSW** organiza los vectores en capas jerárquicas:
- Navega rápido por capas altas y refina en capas bajas, como un GPS.
- Ventaja sobre IVFFlat: no requiere entrenamiento previo y ofrece mejor recall.

**Parámetros importantes**:
- `M=32` → conexiones por nodo (más = mejor calidad, más memoria)
- `efConstruction=200` → esfuerzo al construir
- `efSearch=50` → esfuerzo al buscar vecinos

In [ ]:
# FAISS es una librería de Facebook AI para búsqueda rápida de vectores.
# útil para encontrar chunks semánticamente similares en grandes colecciones.
import faiss

# número de columnas de tu array de embeddings
dim = embeddings.shape[1]
# Convierte cada vector a longitud 1 (vector unitario) usando la norma L2
faiss.normalize_L2(embeddings)

#  creamos el índice hnsw basado en HNSW (Hierarchical Navigable Small World)
# con dimension generada en la linea superior y número de vecinos M en el grafo HNSW (afecta precisión/velocidad)
hnsw = faiss.IndexHNSWFlat(dim, 32)

# controlamos precisión al construir el grafo
hnsw.hnsw.efConstruction = 200
# controlamos precisión en la búsqueda
hnsw.hnsw.efSearch = 50
# Inserta todos los embeddings (4964 vectores) en el índice HNSW
hnsw.add(embeddings)

# imprimos número total de vectores agregados al índice
print("Vectores indexados:", hnsw.ntotal)

Vectores indexados: 4964


### 6. Búsqueda Híbrida: BM25 + FAISS

Para mejorar la recuperación de información se combina:

-**BM25** = búsqueda clásica por palabras clave.

-**FAISS** = búsqueda semántica basada en embeddings (por significado).

La combinación se hace mediante una mezcla lineal controlada por un parámetro alpha:

**score = alpha × FAISS + (1 - alpha) × BM25**

**FAISS** y **BM25** se normalizan entre 0 y 1 para que los scores sean comparables.

- alpha decide la importancia relativa de cada método:

    (i) alpha cercano a 1 → más peso a FAISS (semántica)

    (ii) alpha cercano a 0 → más peso a BM25 (palabras clave)

- Ejemplo típico: score = 0.7 × FAISS + 0.3 × BM25

**Ventajas:**

- Cubre las limitaciones de cada método: FAISS maneja sinónimos y contexto BM25 garantiza coincidencias literales precisas.

- **La función retrieve()** devuelve los k chunks más relevantes según la consulta, integrando semántica y palabras clave de manera eficiente.

In [ ]:
# importando la clase BM25Okapi de la librería rank_bm25
# algoritmo de recuperación de información basado en palabras clave.
from rank_bm25 import BM25Okapi

# Conviertimos cada chunk a lista de palabras minúsculas.
tokenized = [t.lower().split() for t in chunk_texts]
# Inicializamos BM25 con todos los chunks tokenizados (búsquedas por palabras clave)
bm25      = BM25Okapi(tokenized)

# definimos función llamada ** retrieve ** el cual sirve para recuperar los textos más relevantes
# query = la frase o palabra a buscar
# k = número de resultados que quieres devolver
# alpha = peso que mezcla FAISS (semántica) y BM25 (palabras clave)
def retrieve(query, k=10, alpha=0.7):
    # listamos de todos los pedazos de texto
    n = len(chunk_texts)

    # FAISS
    # Convertimos la consulta en un embedding vectorial, Normaliza (similitud coseno.)
    # y Busca en el índice HNSW los k*3 vecinos más cercanos
    q_emb = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    D, I  = hnsw.search(q_emb, min(k*3, n))
    # -------------------------------------

    # Crea un array con ceros para todos los chunks y Asignamos la similitud FAISS solo a los chunks encontrados.
    # por el bucle defindo
    faiss_scores = np.zeros(n)
    for s, i in zip(D[0], I[0]):
        if i >= 0: faiss_scores[i] = float(s)
        # -----------------------------------

    # Escala los scores entre 0 y 1 para que sean comparables con BM25.
    if faiss_scores.max() > 0:
        faiss_scores /= faiss_scores.max()

    # BM25
    # devuelve un score de relevancia para cada chunk según la consulta.
    bm25_raw = bm25.get_scores(query.lower().split())
    # Normaliza BM25 también entre 0 y 1.
    bm25_scores = bm25_raw / bm25_raw.max() if bm25_raw.max() > 0 else bm25_raw

    # Fusión
    # Mezclamos resultados de FAIS y BM25 según un peso alpha.
    # Ordenamos de mayor a menor los chunks más relevantes
    # Se toman solo los k mejores resultado
    hybrid  = alpha * faiss_scores + (1 - alpha) * bm25_scores
    top_idx = np.argsort(hybrid)[::-1][:k]

    # Devuelve una lista: text =el chunk y title = su título original
    return [{"text": chunk_texts[i], "title": chunk_titles[i]} for i in top_idx]

### 7. Reranker Cross-Encoder

FAISS usa un **bi-encoder**: codifica pregunta y documento por separado, luego compara vectores. Rápido pero aproximado.

El reranker usa un **cross-encoder**: analiza pregunta + documento juntos. Mucho más preciso.

Estrategia del pipeline:
1. FAISS recupera top-10 chunks rápido (búsqueda semántica aproximada).
2. El reranker reordena esos top-10 usando cross-encoder para evaluar relevancia exacta.
3. Se seleccionan los top-3 resultados para alimentar al LLM.

**Función rerank():**
- Recibe una consulta y una lista de chunks candidatos.
- Predice un score de relevancia para cada par (consulta, chunk).
- Devuelve los top-k candidatos ordenados por relevancia.

In [ ]:
# Importamos librería de Python diseñada para trabajar con modelos de embeddings de oraciones o textos.
# es un modelo que evalúa la relevancia de un par de textos: (consulta, documento)
from sentence_transformers import CrossEncoder

# Cargamos un modelo pre-entrenado para reranking tipo MS MARCO.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# definimos la funcion que búsca la consulta del usuario y que tenga una lista de resultados previos (chunks)
# devuelve solo los 3 mejores resultados después de reranking
def rerank(query, candidates, top_k=3):
    # Convierte cada candidato en un par (query, texto) y devuelve un score de relevancia
    scores = reranker.predict([(query, c["text"]) for c in candidates])
    # Agrega un nuevo campo "rerank_score" a cada candidato y guarda el score predicho por el CrossEncoder
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    # Retornamos ordenando de mayor a menor relevancia
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

### 8. LLM Qwen (4-bit)
El LLM es el modelo final que genera la respuesta basada en los chunks más relevantes.

**Características:**
- Qwen 1.5 (~1.8B parámetros), cargado con cuantización de 4 bits. Esto reduce su tamaño de ~7GB a ~1GB de VRAM, permitiendo usar GPUs como la T4 de Colab.
- El system prompt le indica que responda **solo con el contexto** proporcionado. Esto es la primera defensa contra alucinaciones.

**Componentes:**
- torch → backend PyTorch para operaciones tensoriales y GPU.
- AutoTokenizer → convierte texto a tokens que el modelo entiende.
- AutoModelForCausalLM → modelo de lenguaje causal capaz de generar texto.
- BitsAndBytesConfig → configuración para carga en 4 bits y ahorro de memoria.

**Flujo en el pipeline:**
1. Recibe los top-k chunks seleccionados por FAISS + Reranker.
2. Genera la respuesta textual final respetando el contexto proporcionado.

In [ ]:
# torch → PyTorch, backend para el modelo.
# AutoTokenizer → convierte texto a tokens que entiende el modelo.
# AutoModelForCausalLM → modelo causal language model, capaz de generar texto.
# BitsAndBytesConfig → configuración para cuantización (usar menos memoria).

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Indica el modelo a cargar: Qwen 1.5 con ~1.8B parámetros
model_name = "Qwen/Qwen1.5-1.8B-Chat"

# Permite cargar el modelo en 4 bits en lugar de 16 o 32. Menos uso de GPU/CPU y acelera inferencia
bnb       = BitsAndBytesConfig(load_in_4bit=True)
# convierte texto ↔ tokens
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# carga los pesos del modelo desde Hugging Face, con Parámetros claves.
model     = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True
)
# Coloca el modelo en modo inferencia, desactivando el entrenamiento
model.eval()
print("✅ Qwen cargado")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

✅ Qwen cargado


### 9. Prompt + generación de respuestas
Este paso utiliza el LLM (Qwen 4-bit) para generar la respuesta final basada en los chunks más relevantes obtenidos del pipeline RAG.

Flujo:
1. Recibe la pregunta del usuario y el contexto (chunks top después de retrieval + reranker).

2. Construye un prompt estructurado:
    - Rol "system": instruye al modelo a usar solo el contexto.
    - Rol "user": incluye la pregunta y el contexto.

3. Convierte el prompt a tokens usando el tokenizer.

4. Genera texto con el modelo Qwen (quantized 4-bit para eficiencia).

5. Devuelve la respuesta textual final.

Importancia:

- Garantiza que el LLM solo use información recuperada (previene alucinaciones).

- Completa la etapa de generación del pipeline RAG.

In [ ]:
# Define una función llamada ask_llm que recibe:
# 1.- context: el texto (chunks) que el LLM puede usar para responder.
# 2.- question: la pregunta que queremos hacerle al LLM.

def ask_llm(context, question):
    # Creamos un prompt estructurado en formato tipo chat (system + user).
    messages = [
        # system → instrucciones generales al LLM
        {"role": "system", "content": "Responde ÚNICAMENTE usando el contexto. Si no está, di 'No encontrado en el contexto'."},
        # user → incluye la pregunta y el contexto real
        {"role": "user",   "content": f"Contexto:\n{context}\n\nPregunta:\n{question}"}
    ]
    # genera un texto listo para el modelo a partir de los messages, aún no convierte el texto en tokens, agrega lo que el modelo espera para iniciar
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Convierte el prompt en tokens que el modelo entiende y devuelve tensores de PyTorch.
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        # Genera la respuesta del LLM:
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)

    # Extrae solo los tokens generados, eliminando los tokens del prompt.
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    # Devuelve la respuesta final del LLM como string.
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### 10. Citation per sentence

Este paso mejora la trazabilidad de la respuesta del LLM, agregando referencias a los chunks originales que respaldan cada oración de la respuesta.

**Flujo:**
1. Divide la respuesta generada en oraciones.

2. Para cada oración, calcula el "overlap" de palabras con cada chunk.

3. Asigna como cita el título del chunk con mayor overlap.

4. Si no hay coincidencia significativa (>10% palabras), marca "no encontrado".

5. Devuelve la respuesta con citas por oración.

**Importancia:**
Permite validar la fuente de cada afirmación.
- Aumenta transparencia y confiabilidad del sistema RAG.
- Útil en contextos académicos o de documentación.

In [ ]:
# Importa la librería re de Python para trabajar con expresiones regulares, usada aquí para dividir texto en oraciones.
import re

# Define una función llamada cite_sentences que recibe:
# answer: la respuesta generada por el LLM.
# chunks: la lista de chunks originales (cada uno con text y title) para buscar coincidencias.
def cite_sentences(answer, chunks):
    # Divide la respuesta en oraciones usando regex:
    sentences = re.split(r'(?<=[.!?])\s+', answer.strip())
    # Filtra oraciones demasiado cortas (menos de 10 caracteres).
    sentences = [s for s in sentences if len(s) > 10]
    # Inicializa una lista vacía donde se guardarán las oraciones con su cita.
    result = []

    # Recorre cada oracion en minúscula,
    # cada título del chunk con mayor coincidencia (inicialmente "no encontrado")
    # y  puntaje de overlap máximo. (inicialmente -1).
    for sent in sentences:
        words = set(sent.lower().split())
        best, best_score = "no encontrado", -1
        # Convierte el texto del chunk a palabras en minúscula,
        # calcula el overlap y si es mayor que best_score, actualiza best_score y best con el título de ese chunk.
        for c in chunks:
            overlap = len(words & set(c["text"].lower().split())) / max(len(words), 1)
            if overlap > best_score:
                best_score, best = overlap, c["title"]
        # Solo considera que un chunk respalda la oración si más del 10% de las palabras coinciden (best_score > 0.1).
        # Si no, marca "no encontrado".
        citation = best if best_score > 0.1 else "no encontrado"
        result.append(f"{sent} [{citation}]")
        # ---------------------------------------------------
    # Devuelve todas las oraciones con citas como un solo string
    return "\n".join(result)

### 11. Hallucination guard + grounding score

Este paso verifica que la respuesta del LLM esté respaldada por el contexto.
Flujo:
1. Extrae todas las palabras de los chunks recuperados.
2. Filtra palabras irrelevantes (stopwords y cortas) de la respuesta.
3. Calcula el grounding score: proporción de palabras de la respuesta presentes en el contexto.
4. Asigna un veredicto según el score:
- ≥0.7 ✅ FUNDAMENTADO
- 0.4–0.7 ⚠️ ADVERTENCIA
- <0.4 ❌ ALUCINACIÓN
5. Devuelve el score y el veredicto.
Importancia:
- Evita respuestas inventadas por el modelo (alucinaciones).
- Garantiza que la información provenga del contenido recuperado.

In [ ]:
# Define la función hallucination_guard que recibe dos parámetros:
# answer: la respuesta generada por el LLM.
# chunks: lista de diccionarios con los fragmentos de texto originales recuperados
def hallucination_guard(answer, chunks):
    # Junta todo el texto de los chunks en un solo string
    ctx_words = set(" ".join(c["text"] for c in chunks).lower().split())
    # Definimos palabras comunes (stopwords) que no aportan significado importante, para ignorarlas en la comparación.
    stopwords = {"el","la","los","las","un","una","es","son","en","de","y","a","the","is","of","and","to","a"}
    # Toma la respuesta answer, la pasa a minúsculas y la divide en palabras.
    content   = [w for w in answer.lower().split() if w not in stopwords and len(w) > 2]
    # filtrado no queda ninguna palabra significativa (content vacío)
    if not content:
        return 0.0, "SIN DATOS"
    # Calcula cuántas palabras relevantes de la respuesta aparecen en el contexto (ctx_words).
    score   = sum(1 for w in content if w in ctx_words) / len(content)
    # Asigna un veredicto según el grounding score
    verdict = "✅ FUNDAMENTADO" if score >= 0.7 else ("⚠️ ADVERTENCIA" if score >= 0.4 else "❌ ALUCINACIÓN")
    # Devuelve el score redondeado a 4 decimales y el veredicto correspondiente.
    return round(score, 4), verdict

### 12. BLEU + ROUGE
Este paso permite medir cuantitativamente la calidad de la respuesta generada comparándola con una referencia “correcta”.

Flujo:
1. Tokeniza y normaliza a minúsculas la respuesta generada y la referencia.
2. Calcula BLEU-1 → mide coincidencia de palabras individuales (unigrams).
3. Calcula ROUGE-L → mide coincidencia de secuencias largas (longest common subsequence).
4. Devuelve un diccionario con BLEU-1 y ROUGE-L redondeados a 4 decimales.

**Importancia:**
- Evalúa la fidelidad de la generación automática.
- Permite comparar distintas estrategias de retrieval, prompts o reranking.
- Complementa el grounding score para medir precisión y coherencia.

In [ ]:
# Importa librerías necesarias:
# nltk → procesamiento de texto y tokenización.
# sentence_bleu → calcula la métrica BLEU (coincidencia de palabras n-gram) a nivel de oración.
# SmoothingFunction → evita problemas cuando no hay coincidencias exactas.
# rouge_scorer → calcula la métrica ROUGE, evaluando similitud entre resumen o texto generado y referencia.

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer


# Descargamos los modelos de tokenización de NLTK.
# para separar frases y palabras (quiet=True es decir no muestra logs de descarga).
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

# Inicializa el scorer de ROUGE:
# "rouge1" → compara coincidencia de unigrams (palabras individuales).
# "rougeL" → compara la longest common subsequence (frases o secuencia de palabras).
rouge    = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
# función para suavizar scores BLEU cuando hay pocas coincidencias, evitando 0 exacto.
smoother = SmoothingFunction().method1

# Definimos la función evaluate que recibe:
# generated → respuesta del LLM.
# reference → respuesta “correcta” o esperada para comparar.
def evaluate(generated, reference):
    ref   = reference.lower().split()
    gen   = generated.lower().split()
    # Calcula BLEU-1: proporción de unigrams de generated que coinciden con reference.
    bleu1 = sentence_bleu([ref], gen, weights=(1,0,0,0), smoothing_function=smoother)
    # Calcula métricas ROUGE entre referencia y texto generado.
    rs    = rouge.score(reference, generated)
    # Devuelve un diccionario con métricas evaluadas:
    return {
        "BLEU-1":  round(bleu1, 4), # proporción de palabras coincidentes.
        "ROUGE-L": round(rs["rougeL"].fmeasure, 4), # proporción de secuencias largas coincidentes.
    }

### 13. Consulta
Aquí el usuario hace la pregunta y se ejecuta todo el pipeline: retrieve → rerank → LLM → hallucination guard.

(i) Se obtiene el contexto de los chunks más relevantes.

(ii) El LLM genera la respuesta usando solo ese contexto.

(iii) Se calcula el grounding score para medir si la respuesta está fundamentada.

**Objetivo:** devolver una respuesta confiable basada en los documentos originales.

In [ ]:
query = "¿Qué es un tablespace en Oracle?"  # ← cambia por tu pregunta (Pregunta del usuario)

# Recuperación: se buscan los chunks más relevantes (FAISS + BM25)
docs    = retrieve(query, k=10)
# Reranking: top-10 se reordena con Cross-Encoder, quedando top-3
top     = rerank(query, docs, top_k=3)
# Contexto: se unen los textos de los chunks rerankeados
context = "\n".join(c["text"] for c in top)
# Generación: el LLM Qwen produce la respuesta usando solo este contexto
answer  = ask_llm(context, query)
# Grounding: se evalúa si la respuesta está fundamentada en los chunks
score, verdict = hallucination_guard(answer, top)

# Salida: se imprime la respuesta y su grounding score
print("RESPUESTA:\n", answer)
print("\nGrounding score:", score, verdict)

RESPUESTA:
 Un tablespace en Oracle es una unidad lógica básica de memoria específica para almacenar tablas relacionadas entre sí. Se utiliza para almacenar y organizar los datos de manera segura y organizada en un sistema de gestión de recursos de Oracle.
Una tabla es una estructura de datos en Oracle que contiene filas y columnas, cada una con un nombre específico y un tipo de datos determinado. Las columnas pueden tener diferentes tipos de datos, incluyendo valores de booleanos, números, strings, arrays, y otros tipos de datos específicos. Además, las columnas pueden tener restricciones sobre su tamaño y forma, lo que permite garantizar que los datos se mantengan organizados y seguros durante la ejecución del programa.
Un tablespace se crea mediante la operación "CREATE TABLESPACE" en el entorno de Oracle y especificando un nombre y un espacio de tablas. El espacio de tablas puede ser compartido entre varias tablas

Grounding score: 0.2917 ❌ ALUCINACIÓN


**concluimos lo siguiente:**

El grounding score de 0.2917 indica que menos del 30% de las palabras clave de la respuesta coinciden con los chunks seleccionados del contexto.

El veredicto ❌ ALUCINACIÓN significa que la respuesta contiene información no fundamentada en los documentos que proporcionaste.

Aunque la respuesta suena correcta sobre Oracle, tu pipeline detectó que gran parte del contenido no estaba respaldado por los documentos indexados, lo que puede generar errores o imprecisiones.

### Varias consultas

In [ ]:
questions = [
    ("¿Cuáles son las tareas básicas de un DBA?",     "El DBA debe instalar software, monitorear el sistema y garantizar la seguridad."),
    ("¿Qué es un índice en Oracle?",                  "Un índice es una estructura que permite recuperar datos de forma más rápida."),
    ("What is linear algebra?", "Linear algebra studies vectors, matrices and linear transformations."),
]

results = []

for query, reference in questions:
    docs    = retrieve(query, k=10)
    top     = rerank(query, docs, top_k=3)
    context = "\n".join(c["text"] for c in top)
    answer  = ask_llm(context, query)
    cited   = cite_sentences(answer, top)
    score, verdict = hallucination_guard(answer, top)
    metrics = {"grounding": score, "verdict": verdict}
    metrics.update(evaluate(answer, reference))
    results.append({"query": query, "answer": answer, "cited": cited, "metrics": metrics})

    print("="*60)
    print("Q:", query)
    print("\nRespuesta con citas:")
    print(cited)
    print("\nMétricas:", metrics)

Q: ¿Cuáles son las tareas básicas de un DBA?

Respuesta con citas:
Las tareas básicas del DBA incluyen:

1. [oracle-introduccion · p21]
Creación de tables: El DBA debe crear cuestiones de usuario y asignarles tablespaces diferentes para almacenar los datos en un formato diferente. [oracle-introduccion · p21]
Esto puede incluir la creación de tablas de usuario, que permiten a los usuarios acceder a los datos en función de su identificación, y la creación de tablas de espacio, que permiten a los usuarios acceder a los datos en función de su acceso. [oracle-introduccion · p21]
Configuración de columnas: El DBA debe configurar las columnas de cada tabla de usuario para garantizar que los datos sean únicos y que no puedan ser duplicados. [oracle-introduccion · p21]
También debe establecer restricciones sobre la cantidad de valores permitidos en las columnas y la tamaño máximo de los datos permitidos en cada columna. [oracle-introduccion · p8]
Control de acceso: El DBA debe controlar el acce

### Reporte final

In [ ]:
print(f"{'='*65}")
print(f"{'Pregunta':<30} {'Grounding':>10} {'BLEU-1':>8} {'ROUGE-L':>8}  Veredicto")
print("-"*65)
for r in results:
    q  = r['query'][:28] + ".." if len(r['query']) > 28 else r['query']
    g  = r['metrics']['grounding']
    b1 = r['metrics'].get('BLEU-1', '-')
    rl = r['metrics'].get('ROUGE-L', '-')
    v  = r['metrics']['verdict']
    print(f"{q:<30} {g:>10.4f} {str(b1):>8} {str(rl):>8}  {v}")
print("="*65)

Pregunta                        Grounding   BLEU-1  ROUGE-L  Veredicto
-----------------------------------------------------------------
¿Cuáles son las tareas básic..     0.4023   0.0552    0.071  ⚠️ ADVERTENCIA
¿Qué es un índice en Oracle?       0.4476   0.0533    0.104  ⚠️ ADVERTENCIA
What is linear algebra?            0.0192   0.0144   0.0248  ❌ ALUCINACIÓN


1️⃣ Consulta: “¿Cuáles son las tareas básicas de un DBA?”

* Respuesta con citas: La LLM generó información parcialmente relacionada con tus documentos (oracle-introduccion · p21/p8).

* Grounding score: 0.4023 → ⚠️ ADVERTENCIA. Solo ~40% de las palabras importantes de la respuesta están respaldadas por el contexto.

* BLEU / ROUGE: Muy bajos (0.05–0.07), indicando que la respuesta no coincide literalmente con la referencia.

* Interpretación: La respuesta contiene información aproximada, parcialmente fundamentada, pero no se puede considerar completamente confiable.

2️⃣ Consulta: “¿Qué es un índice en Oracle?”

* Respuesta con citas: Muchos chunks están citados correctamente, pero algunas afirmaciones no coinciden exactamente con los documentos.

* Grounding score: 0.4476 → ⚠️ ADVERTENCIA, nuevamente evidencia parcial de fundamentación.

* BLEU / ROUGE: 0.05–0.10 → baja coincidencia literal.

* Interpretación: Aunque el contenido es relevante, todavía hay riesgo de errores o imprecisiones si se confía solo en la generación.

3️⃣ Consulta: “What is linear algebra?”

* Respuesta con citas: Casi todas las oraciones no se encuentran en los chunks (no encontrado).

* Grounding score: 0.0192 → ❌ ALUCINACIÓN.

* BLEU / ROUGE: Extremadamente bajos, la respuesta no coincide con la referencia.

* Interpretación: El modelo alucina completamente sobre un tema que no tiene cobertura en tu dataset. Esto evidencia la importancia de que el dataset incluya documentos relevantes para cada consulta.